## install

In [1]:
!pip install biopython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 28.9 MB/s eta 0:00:00


In [2]:
!apt-get install -y bedtools

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  bedtools
0 upgraded, 1 newly installed, 0 to remove and 35 not upgraded.
Need to get 563 kB of archives.
After this operation, 1,548 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 bedtools amd64 2.30.0+dfsg-2ubuntu0.1 [563 kB]
Fetched 563 kB in 1s (685 kB/s)
Selecting previously unselected package bedtools.
(Reading database ... 126281 files and directories currently installed.)
Preparing to unpack .../bedtools_2.30.0+dfsg-2ubuntu0.1_amd64.deb ...
Unpacking bedtools (2.30.0+dfsg-2ubuntu0.1) ...
Setting up bedtools (2.30.0+dfsg-2ubuntu0.1) ...


In [3]:
!apt-get install -y samtools


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libhts3 libhtscodecs2
Suggested packages:
  cwltool
The following NEW packages will be installed:
  libhts3 libhtscodecs2 samtools
0 upgraded, 3 newly installed, 0 to remove and 35 not upgraded.
Need to get 963 kB of archives.
After this operation, 2,270 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libhtscodecs2 amd64 1.1.1-3 [53.2 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libhts3 amd64 1.13+ds-2build1 [390 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 samtools amd64 1.13-4 [520 kB]
Fetched 963 kB in 1s (1,025 kB/s)
Selecting previously unselected package libhtscodecs2:amd64.
(Reading database ... 126334 files and directories currently installed.)
Preparing to unpack .../libhtscodecs2_1.1.1-3_amd64.deb ...
Unpacking libhtscodecs2:amd64 (1.1.

In [4]:
from Bio import SeqIO
import pandas as pd

In [5]:
from google.colab import files
uploaded = files.upload()

## VIS_positives

In [6]:
positive_count = sum(1 for _ in SeqIO.parse("HPV.fa", "fasta"))
print("Number of positive sequences:", positive_count)

Number of positive sequences: 69079


In [7]:


bed_lines = []

for record in SeqIO.parse("HPV.fa", "fasta"):
    header = record.id  # e.g., chr9:99841705-99842705
    chrom, coords = header.split(":")
    start, end = coords.split("-")
    bed_lines.append(f"{chrom}\t{start}\t{end}")

# Save to BED file
with open("HPV_positives.bed", "w") as f:
    f.write("\n".join(bed_lines))


In [8]:
!bedtools sort -i HPV_positives.bed > HPV_positives_sorted.bed

In [9]:
!head HPV_positives_sorted.bed

chr1	513	1513
chr1	789	1789
chr1	9524	10524
chr1	9610	10610
chr1	9664	10664
chr1	9685	10685
chr1	9770	10770
chr1	9868	10868
chr1	12617	13617
chr1	12617	13617


In [10]:
!head HPV_positives.bed

chr9	99841705	99842705
chr3	169299140	169300140
chr7	5276228	5277228
chr7	5276189	5277189
chr3	195825139	195826139
chr1	38001640	38002640
chr1	38001638	38002638
chr1	38001636	38002636
chr1	38001635	38002635
chr1	38001591	38002591


## Gencode fasta

In [11]:
!wget https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_48/GRCh38.p14.genome.fa.gz
!gunzip /content/GRCh38.p14.genome.fa.gz
!samtools faidx GRCh38.p14.genome.fa

--2025-07-17 14:13:53--  https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_48/GRCh38.p14.genome.fa.gz
Resolving ftp.ebi.ac.uk (ftp.ebi.ac.uk)... 193.62.193.165
Connecting to ftp.ebi.ac.uk (ftp.ebi.ac.uk)|193.62.193.165|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 897322567 (856M) [application/x-gzip]
Saving to: ‘GRCh38.p14.genome.fa.gz’

GRCh38.p14.genome.f 100%[===================>] 855.75M  3.20MB/s    in 88s     

2025-07-17 14:15:22 (9.71 MB/s) - ‘GRCh38.p14.genome.fa.gz’ saved [897322567/897322567]



In [12]:
# Create the genome size file from the index
!cut -f1,2 GRCh38.p14.genome.fa.fai > hg38.chrom.sizes
# Sort it (required by bedtools complement)
!sort -k1,1 -k2,2n hg38.chrom.sizes > hg38.chrom.sorted.sizes

## complement, random

In [35]:
N = 2000

In [36]:
!bedtools slop -i HPV_positives_sorted.bed -g hg38.chrom.sorted.sizes -b $N > HPV_extended.bed

# 3. Get the complement of the extended regions (available regions for negatives)
!bedtools complement -i HPV_extended.bed -g hg38.chrom.sorted.sizes > uncovered_regions.bed

In [37]:
# 2. Sample 10x negatives
!bedtools random -g hg38.chrom.sizes -l 1000 -n 200000 > all_negatives.bed

# 3. Subtract positives to avoid overlap
!bedtools subtract -a all_negatives.bed -b HPV_positives_sorted.bed > negatives.bed

***** WARNING: File all_negatives.bed has inconsistent naming convention for record:
ML143368.1	141715	142715	5	1000	-

***** WARNING: File all_negatives.bed has inconsistent naming convention for record:
ML143368.1	141715	142715	5	1000	-



In [38]:
!bedtools intersect -a negatives.bed -b HPV_positives_sorted.bed -u


***** WARNING: File negatives.bed has inconsistent naming convention for record:
ML143368.1	141715	142715	5	1000	-

***** WARNING: File negatives.bed has inconsistent naming convention for record:
ML143368.1	141715	142715	5	1000	-



In [39]:
!bedtools getfasta -fi GRCh38.p14.genome.fa -bed negatives.bed -fo negative_HPV.fasta

In [40]:
!head negative_HPV.fasta

>chr4:137621717-137622717
TGATACATGTATACAATGTGTAATGATTAAATCAGAGTAATTGAACTATTCGTCACCTTAAACATTTATCTTTTCTTTGTGCTGGGAACACTACATTTTTTCTCTTCTAGCTACTTTGAAATACATAGTAAATTATTGTTACCTATAATTTTCCTACTATACTATCAAACAGTAGAACTTGTTCCTTCTACCTGACTATATTTTTGTGTCCAAAGACCATATTCTTTTTTCCCTTTCTTCTAAAGAAAAATTGTACATACTTACACAAATTAACTCATGAAAGAAAAATATCTGGATAATACAAGAGGATACAGGCAGGACTGTGAGGATTCTCTGTGATAATAACAAGTTAACTGCAGTTTCTAAGACTATGTCTTTGAACATTTATTATCCATTATTATGCTATTCAAGAAAAGGAGCACCTCCTCACTGTTTACCTTCTTCCCTATTTCTTGTTTTTCCTGTAGTTGTTACCCCTCTTCTATGCCTGGATATGCATGATTTTTTCAGTGGAGAATTTCTGCAAAGCAGACACAACATTTTTATTAGTGAAGTTAACACTATGCTAGAAGAATAGTAAATTCAAGCTATAAATCCACCAGCTTTCAGGAGACTCATTTATTCTGTTCCATATGAAATAAAAATCTACTACAGGGGAAGAGTTGAATATGACCATATACATTTATGAAGGAACATTTTACACTTTACTGCTGTAGGCTACTACTGGCCTTAGGAGCTGATGTTCATTTTATTTGATTCAAAATAAAAGTACAATAAAAAATAAAGCAGACATCCCCCACCACTCAATCGCCCACCCACCCACCCACCCAACAAGGTATCTTGGATCACAAAATGCCAGTCCCCTGTGGTTATTTAAAGAAACCTGGGCAGGTTTCCAAGTTCTGGTGTCTGTGACCTATTACACGGAGGCAACAATAATAGTTCCTCATGAAATAAATCAATGGACATACGTA

In [41]:
positive_count = sum(1 for _ in SeqIO.parse("negative_HPV.fasta", "fasta"))
print("Number of neg sequences:", positive_count)

Number of neg sequences: 199910


## filter for only "A" "C" "T" and "G"

In [42]:

input_file = "HPV.fa"
output_file = "HPV_clean.fa"

# Valid DNA characters
valid_bases = set("ACTG")

# Filter and write clean sequences
with open(output_file, "w") as out_f:
    for record in SeqIO.parse(input_file, "fasta"):
        seq = str(record.seq).upper()
        if set(seq).issubset(valid_bases):
            SeqIO.write(record, out_f, "fasta")

print("Filtering complete. Clean sequences saved to:", output_file)


Filtering complete. Clean sequences saved to: HPV_clean.fa


In [43]:
positive_count = sum(1 for _ in SeqIO.parse("HPV_clean.fa", "fasta"))
print("Number of positive sequences:", positive_count)

Number of positive sequences: 68865


In [44]:
input_file = "negative_HPV.fasta"
output_file = "negative_HPV_clean.fa"

with open(output_file, "w") as out_f:
    for record in SeqIO.parse(input_file, "fasta"):
        seq = str(record.seq).upper()
        if set(seq).issubset(valid_bases):
            SeqIO.write(record, out_f, "fasta")

print("Filtering complete. Clean sequences saved to:", output_file)

Filtering complete. Clean sequences saved to: negative_HPV_clean.fa


In [45]:
neg_count = sum(1 for _ in SeqIO.parse("negative_HPV_clean.fa", "fasta"))
print("Number of negative sequences:", neg_count)

Number of negative sequences: 190032


In [46]:
from google.colab import files
files.download('HPV_clean.fa')
files.download('negative_HPV_clean.fa')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>